### Story 618 — Perform basic geometry checks when adding/updating items in the catalog
  * https://pforge-exchange2.astrium.eads.net/jira/browse/RSPY-618

This story adds **server-side validation and enforcement** for `geometry` and `bbox` when **creating or updating STAC Items** via:

- `POST  /catalog/collections/{ownerId:collectionId}/items`
- `PUT   /catalog/collections/{ownerId:collectionId}/items/{featureId}`
- `PATCH /catalog/collections/{ownerId:collectionId}/items/{featureId}`

It implements two ESA/STAC requirements:

- **STAC-CORE-ITEM-REQ-0220 — Polygon Geometry**: Polygon/MultiPolygon rings must follow the **right-hand rule** (exterior ring **CCW**, interior rings **CW**), per RFC 7946.
- **STAC-CORE-ITEM-REQ-0230 — Minimum-bounding rectangle**: `bbox` must be **2D** and formatted as **4 values**: `[minLon, minLat, maxLon, maxLat]` (**SW → NE** order).

#### Accepted cases

- **`geometry = null` AND `bbox = null`/missing**  
  Accepted (e.g., CADIP sessions). **No geometry checks performed**.

#### Auto-enforcement (server modifies the item)

- **`geometry` present AND `bbox` missing**  
  The server **computes** `bbox` from the geometry bounds and adds it.

- **`geometry` present AND `bbox` present**  
  The server enforces that `bbox` is **consistent with geometry bounds** (strict mode). If consistent, it normalizes numbers to floats.

#### Rejected cases (HTTP 400 + clear message; item is NOT added/updated)

- `geometry = null` but `bbox` is provided →  
  `"Invalid STAC item: bbox provided but geometry is null."`

- `geometry` is not a GeoJSON object (`dict`) →  
  `"Invalid GeoJSON geometry: expected an object."`

- `geometry` cannot be parsed by Shapely (`shape(...)` fails) →  
  `"Invalid GeoJSON geometry: {exception_message}"`

- `geometry` is empty →  
  `"Invalid GeoJSON geometry: empty geometry is not allowed."`

- `geometry` is topologically invalid (`is_valid == False`, e.g., self-intersection) →  
  `"Invalid GeoJSON geometry: {shapely_reason}."`

- `MultiPolygon` has invalid `coordinates` type →  
  `"Invalid GeoJSON MultiPolygon: coordinates must be an array."`

- Polygon/MultiPolygon ring structure / validity errors:

  - No rings / rings not an array →  
    `"Invalid {Polygon|MultiPolygon[i]}: expected at least one linear ring."`

  - Ring is not an array →  
    `"Invalid {ring_label}: ring must be an array of positions."`

  - Ring has < 4 positions →  
    `"Invalid {ring_label}: ring must contain at least 4 positions."`

  - Ring is not closed (first != last) →  
    `"Invalid {ring_label}: ring must be closed (first and last positions must match)."`

  - Degenerate ring (area = 0) →  
    `"Invalid {ring_label}: degenerate ring area is zero."`

  - Wrong orientation (right-hand rule) →  
    `"Invalid {ring_label}: expected {counterclockwise|clockwise} orientation (right-hand rule)."`

  - A position is not at least `[lon, lat]` →  
    `"Invalid {position_label}: expected at least [lon, lat]."`

- `bbox` validation errors (REQ-0230):

  - `bbox` not an array →  
    `"Invalid bbox: expected an array."`

  - `bbox` length != 4 →  
    `"Invalid bbox: expected an array of length 4 [minLon, minLat, maxLon, maxLat]."`

  - `bbox` SW/NE order invalid (`minLon > maxLon` or `minLat > maxLat`) →  
    `"Invalid bbox: expected southwesterly point followed by northeasterly point."`

  - `bbox` contains non-numeric / boolean values →  
    `"Invalid numeric value in bbox."`

  - `bbox` does not match geometry bounds (strict consistency) →  
    `"Inconsistent bbox for geometry. Expected {expected_bbox}, got {parsed_bbox}."`

In [ ]:
# Init environment before running a demo notebook.
from resources.utils import *  
import pprint, copy, pystac
init_demo()
# Reload the global vars again
from resources.utils import * 

from resources.dask_clusters.dask_main_env import *
await init_dask_cluster_staging()

pp = pprint.PrettyPrinter(indent=2, width=80, sort_dicts=False, compact=True)

owner = catalog_client.owner_id

In [ ]:
# Create a test collection
CATALOG_COLLECTION_ID = "SPRINT_34_RSPY_618_TEST_COLLECTION"
collection = create_test_collection(CATALOG_COLLECTION_ID)
items = catalog_client.get_items(CATALOG_COLLECTION_ID)
list(items)

In [ ]:
cadip_no_geometry_no_bbox = cadip_client.search(
    method="GET", 
    stac_filter="externalIds='cadip:6f3c8d91-2b0e-492d-aef6-87b24f2bcb1e'")
cadip_no_geometry_no_bbox

### Why this notebook uses raw HTTP calls

`http_session.post/put/patch` targets the Catalog endpoints directly.  
`catalog_client.add_item/update_item` adds an extra client-side validation layer (`pystac.Item.validate()`), which can fail before the request is sent.

Using raw HTTP keeps the demo focused on Story 618 endpoint requirements:
- `POST/PUT/PATCH` status codes and responses
- Catalog-side error messages
- persistence outcome (item added/updated or rejected)


In [ ]:
# 1.
# Insert a CADIP session STAC item without geometry nor bbox. Check that the item is successfully inserted.
item_dict = cadip_no_geometry_no_bbox.to_dict()["features"][0]
item_dict["collection"] = CATALOG_COLLECTION_ID
item_dict.setdefault("stac_extensions", [])
item_dict["assets"] = {}  # Avoid triggering the bucket-transfer logic (the temp S3 copy/move of assets)

# POST  /catalog/collections/{ownerId:collectionId}/items
url = f"{catalog_client.href_service}/catalog/collections/{owner}:{CATALOG_COLLECTION_ID}/items"
if cluster_mode:
    resp = http_session.post(url, json=item_dict, headers={"x-api-key": os.environ["RSPY_APIKEY"]}, timeout=120)
else:
    resp = http_session.post(url, json=item_dict, timeout=120)

assert resp.status_code == 201
assert resp.json()['bbox'] == None
assert resp.json()['geometry'] == None

result = list(catalog_client.get_collection(CATALOG_COLLECTION_ID).get_items())
result[0]

In [ ]:
item_collection_prip = prip_client.search(
    method='GET',
    stac_filter="externalIds='prip:b99c8f80-ee84-4854-bd36-15b18ac0ecca'")
item_collection_prip

In [ ]:
# 2.
# Insert an item with an invalid geojson geometry.
# Check that HTTP 400 bad request error is returned with a clear error message and that the item is NOT added to the catalog.
# (a) - invalid geojson geometry (not an object)

item_dict = copy.deepcopy(item_collection_prip.to_dict()['features'][0])
item_dict["collection"] = CATALOG_COLLECTION_ID
# invalid geojson geometry (not an object)
item_dict["geometry"] = "not-an-object"
item_dict["assets"] = {}  # Avoid triggering the bucket-transfer logic (the temp S3 copy/move of assets)

# POST  /catalog/collections/{ownerId:collectionId}/items
url = f"{catalog_client.href_service}/catalog/collections/{owner}:{CATALOG_COLLECTION_ID}/items"
if cluster_mode:
    resp = http_session.post(url, json=item_dict, headers={"x-api-key": os.environ["RSPY_APIKEY"]}, timeout=120)
else:
    resp = http_session.post(url, json=item_dict, timeout=120)

assert resp.status_code == 400
pp.pprint(resp.json())

In [ ]:
# 2. 
# Insert an item with an invalid geojson geometry. 
# Check that HTTP 400 bad request error is returned with a clear error message and that the item is NOT added to the catalog.
# (b) - invalid geojson geometry (ring not closed)

item_dict = copy.deepcopy(item_collection_prip.to_dict()['features'][0])
item_dict["collection"] = CATALOG_COLLECTION_ID
# invalid geojson geometry (ring not closed)
item_dict["geometry"] = {"type": "Polygon", "coordinates": [[[0,0],[1,0],[1,1],[0,1]]]}
item_dict["assets"] = {}  # Avoid triggering the bucket-transfer logic (the temp S3 copy/move of assets)

# POST  /catalog/collections/{ownerId:collectionId}/items
url = f"{catalog_client.href_service}/catalog/collections/{owner}:{CATALOG_COLLECTION_ID}/items"
if cluster_mode:
    resp = http_session.post(url, json=item_dict, headers={"x-api-key": os.environ["RSPY_APIKEY"]}, timeout=120)
else:
    resp = http_session.post(url, json=item_dict, timeout=120)

assert resp.status_code == 400
pp.pprint(resp.json())

In [ ]:
# 3.
# Insert an item with a valid geojson geometry but no bbox.
# Check that RS-Server computes and adds the bbox as per STAC-CORE-ITEM-REQ-0230 requirement.

item_dict = copy.deepcopy(item_collection_prip.to_dict()['features'][0])
item_dict["collection"] = CATALOG_COLLECTION_ID
# invalid geojson geometry - no bbox - RS-Server computes and adds the bbox 
item_dict.pop("bbox", None)  # no bbox
item_dict["assets"] = {}  # Avoid triggering the bucket-transfer logic (the temp S3 copy/move of assets)

# POST  /catalog/collections/{ownerId:collectionId}/items
url = f"{catalog_client.href_service}/catalog/collections/{owner}:{CATALOG_COLLECTION_ID}/items"
if cluster_mode:
    resp = http_session.post(url, json=item_dict, headers={"x-api-key": os.environ["RSPY_APIKEY"]}, timeout=120)
else:
    resp = http_session.post(url, json=item_dict, timeout=120)

assert resp.status_code == 201
pp.pprint(resp.json())

In [ ]:
# 2 items should be present in catalog
result = list(catalog_client.get_collection(CATALOG_COLLECTION_ID).get_items())
assert len(result) == 2
ItemCollection(result)

In [ ]:
# 4.
# Try to replace the contents of a valid item with an invalid geojson geometry using PUT. 
# Check that HTTP 400 bad request error is returned with a clear error message and that the item is NOT modified in the catalog.

item_id = "S2B_OPER_MSI_L0__GR_2BPS_20250801T074015_S20250801T070620_D04_N05.11"
original_item = catalog_client.get_item(CATALOG_COLLECTION_ID, item_id).to_dict()

# invalid GeoJSON - not counterclockwise (CCW) orientation
bad_item = copy.deepcopy(original_item)
ring = original_item["geometry"]["coordinates"][0]
rev = ring[:-1][::-1]
bad_item["geometry"] = {"type": "Polygon", "coordinates": [rev + [rev[0]]]}

# PUT (replace)
url = f"{catalog_client.href_service}/catalog/collections/{owner}:{CATALOG_COLLECTION_ID}/items/{item_id}"
if cluster_mode:
    resp = http_session.put(url, json=bad_item, headers={"x-api-key": os.environ["RSPY_APIKEY"]}, timeout=120)
else:
    resp = http_session.put(url, json=bad_item, timeout=120)

assert resp.status_code == 400
pp.pprint(resp.json())

# not modified in catalog
after_item = catalog_client.get_item(CATALOG_COLLECTION_ID, item_id).to_dict()
assert after_item["geometry"] == original_item["geometry"]
assert after_item.get("bbox") == original_item.get("bbox")

In [ ]:
# 5. Same as above with PATCH.
# Try to replace the contents of a valid item with an invalid geojson geometry using PATCH.
# Check that HTTP 400 bad request error is returned with a clear error message and that the item is NOT modified in the catalog.

# empty geometry
payload = {"geometry": {"type": "Polygon", "coordinates": []}, "properties": {}}

if cluster_mode:
    resp = http_session.patch(url, json=payload, headers={"x-api-key": os.environ["RSPY_APIKEY"]}, timeout=120)
else:
    resp = http_session.patch(url, json=payload, timeout=120)

assert resp.status_code == 400
pp.pprint(resp.json())

# not modified in catalog
after_item = catalog_client.get_item(CATALOG_COLLECTION_ID, item_id).to_dict()
assert after_item["geometry"] == original_item["geometry"]
assert after_item.get("bbox") == original_item.get("bbox")

In [ ]:
# 6. 
# Try to remove bbox of a valid item using PUT. 
# Check that RS-Server computes again the bbox and adds it back, so that the item in catalog still has a bbox.

url = f"{catalog_client.href_service}/catalog/collections/{owner}:{CATALOG_COLLECTION_ID}/items/{item_id}"
original_item = catalog_client.get_item(CATALOG_COLLECTION_ID, item_id).to_dict()

# PUT - with bbox removed
modified_item = copy.deepcopy(original_item)
modified_item.pop("bbox", None)

if cluster_mode:
    resp = http_session.put(url, json=modified_item, headers={"x-api-key": os.environ["RSPY_APIKEY"]}, timeout=120)
else:
    resp = http_session.put(url, json=modified_item, timeout=120)

assert resp.status_code == 200
pp.pprint(resp.json())

# bbox was re-computed
after_item = catalog_client.get_item(CATALOG_COLLECTION_ID, item_id).to_dict()
assert after_item["bbox"] is not None
assert after_item["geometry"] == original_item["geometry"]
assert after_item.get("bbox") == original_item.get("bbox")


In [ ]:
# 7. Same as above with PATCH. 
# Try to remove bbox of a valid item using PATCH.
# Check that RS-Server computes again the bbox and adds it back, so that the item in catalog still has a bbox.

url = f"{catalog_client.href_service}/catalog/collections/{owner}:{CATALOG_COLLECTION_ID}/items/{item_id}"

# PATCH: remove bbox (middleware recomputes from geometry)
payload = {"bbox": None, "properties": {}}

if cluster_mode:
    resp = http_session.patch(url, json=payload, headers={"x-api-key": os.environ["RSPY_APIKEY"]}, timeout=120)
else:
    resp = http_session.patch(url, json=payload, timeout=120)

assert resp.status_code == 200
pp.pprint(resp.json())

# bbox was re-computed
after_item = catalog_client.get_item(CATALOG_COLLECTION_ID, item_id).to_dict()
assert after_item["bbox"] is not None
assert after_item["geometry"] == original_item["geometry"]
assert after_item.get("bbox") == original_item.get("bbox")

In [ ]:
# 2 items should be present in catalog
result = list(catalog_client.get_collection(CATALOG_COLLECTION_ID).get_items())
assert len(result) == 2
ItemCollection(result)

#### RSPY-979 Return valid geojson geometries from PRIP/LTA

The PRIP station returns the original GeoFootprint, while rs-server-prip now reorients the polygon to return valid RFC 7946 GeoJSON. 
The goal is to show that the geometry is fixed at PRIP/STAC level, so the item can be staged to the Catalog without error.

In [ ]:
import requests
import json
if cluster_mode:
    prip_base_url = "http://mockup-prip-s2b.processing.svc.cluster.local:8080"
else:
    prip_base_url = "http://prip-station:5000"

# Get a mock PRIP access token to query the station directly.
url = f"{prip_base_url}/oauth2/token"
payload = 'client_id=client_id&client_secret=client_secret&username=test&password=test&grant_type=password'
headers = {
  'Content-Type': 'application/x-www-form-urlencoded'
}

response = requests.request("POST", url, headers=headers, data=payload)
token = json.loads(response.text)['access_token']

# Fetch the raw OData product from the PRIP station, before rs-server STAC conversion.
url = f"{prip_base_url}/Products?$filter=Id%20eq%2005fa4a7e-4a99-4c92-9f0b-5dbb2a5fbe55"

payload = {}
headers = {
  'Authorization': f'Bearer {token}'
}

response = requests.request("GET", url, headers=headers, data=payload)
item_prip_station = response.json()['value'][0]

# Fetch the same product through rs-server-prip to compare the returned STAC geometry.
item_rs_server_prip = prip_client.search(
    method='GET',
    stac_filter="externalIds='prip:05fa4a7e-4a99-4c92-9f0b-5dbb2a5fbe55'")

# Compare the original station footprint with the geometry normalized by rs-server-prip.
g1 = item_prip_station['GeoFootprint']['coordinates'][0]
g2 = item_rs_server_prip.to_dict()['features'][0]['geometry']['coordinates'][0]

moved = [p for i, p in enumerate(g2[:-1]) if g1.index(p) != i]
colors = {tuple(moved[0]): '\033[93m', tuple(moved[1]): '\033[94m'}
reset = '\033[0m'

print(item_rs_server_prip.to_dict()['features'][0]['id'])
for title, lst in [('prip-station geometry', g1), ('rs-server-prip fixed geometry', g2)]:
    print(title)
    for p in lst:
        c = colors.get(tuple(p), '')
        print(f'{c}{p}{reset if c else ""}')
    print()

In [ ]:
# Stage the rs-server-prip item to confirm the fixed geometry is accepted by the Catalog.
resp = staging_client.run_staging(item_rs_server_prip.to_dict(), CATALOG_COLLECTION_ID)
staging_client.wait_for_jobs(resp, logger)

In [ ]:
result = list(catalog_client.get_collection(CATALOG_COLLECTION_ID).get_items())
assert len(result) == 3
ItemCollection(result)

In [ ]:
# Fetch three PRIP mock products whose footprints cover the geometry cases under validation:
# LineString crossing the antimeridian, invalid antimeridian polygon, and valid antimeridian polygon.
geometry_test_ids = [
    "4db05e5e-16d7-4c15-8ca1-9a7d31d06eba",
    "cf978bc8-0588-43da-8f40-7717d1b85fae",
    "d2a1b7f8-9c34-4e1a-8aef-123456789abc",
]

items_to_stage = prip_client.search(
    method="GET",
    stac_filter="externalIds='prip:" + ", ".join(geometry_test_ids) + "'",
)

assert len(items_to_stage) == 3
ItemCollection(items_to_stage)

In [ ]:
items_id = [item.id for item in items_to_stage]

staged_items = stage_data(
    items_to_stage.to_dict(),
    items_id=items_id,
    catalog_collection_name=CATALOG_COLLECTION_ID,
)

assert len(staged_items) == 3
ItemCollection(staged_items)

In [ ]:
result = catalog_client.remove_collection(CATALOG_COLLECTION_ID)
assert result.json()["deleted collection"] == CATALOG_COLLECTION_ID
pp.pprint(result.json())